# Knowledge Agent: Hierarchical Routing Tree (SQL Base)
Dieses Jupyter Notebook initialisiert und konfiguriert das relationale Datenfundament für den Knowledge Agent.

Zweck der Architektur:
- Hierarchisches Routing (State-Graph / Decision Tree): Der Agent nutzt diese SQLite-Datenbank wie einen strukturierten Entscheidungsbaum (analog zu einem Random Forest), um eingehende Anfragen schrittweise zu analysieren, über logische Verzweigungen (Knoten) den optimalen Ausführungspfad zu bestimmen und mittels der Verhaltensreihenfolgetabelle abzuarbeiten.

- Dynamische Tabellenverknüpfung & Rücksprung-Logik: Bei spezifischen Anforderungen (wie UI-Validierungen in separaten Fach-Tabellen) verknüpft sich der Agent zielgerichtet mit diesen Datensätzen, führt die Prüfung aus und kehrt anschließend automatisch in den übergeordneten Ablauf zurück.

- Selbstdokumentation & Erfahrungsspeicher: Die Struktur ist so ausgelegt, dass das LLM über das System-Manifest und den Ausführungsspeicher (agent_execution_memory) seine eigenen Navigationsregeln ausliest, vergangene Problemlösungen gewichtet und den Baum dynamisch erweitert.

## Architektur-Übersicht der Datenbank
Die Datenbank gliedert sich in zentrale Komponenten, die als Wissensbasis und Steuerungslogik für den Agenten dienen, um vergangene Problemlösungen abzurufen und dynamische Verknüpfungen herzustellen:

1. agent_system_manifest: Enthält die grundlegenden Systemanweisungen und Direktiven. Das LLM liest diese Manifestdatei als allererstes aus, um zu verstehen, wie es den Routing-Baum zu navigieren hat.

2. agent_routing_tree_index: Bildet die oberste Spitze der Pyramide. Sie teilt sich in die jeweiligen Haupt- und Spezialisierungsstränge auf (z. B. Architektur, Data Analytics, Coding oder Vision).

3. agent_behavior_sequence (Verhaltensreihenfolgetabelle): Verbindet die jeweiligen Knotenpunkte und steuert die genaue Abarbeitungsreihenfolge der Schritte (A -> B -> C).

- Beispiel für die dynamische Verknüpfung: Wenn der Agent im Strang „Architektur“ auf eine spezifische Aufgabe stößt (wie die Validierung eines UI-Elements), enthält die Verhaltensreihenfolge den Verweis: „Führe die Tabelle StartButton aus“.

- Der Agent verknüpft sich daraufhin direkt mit der StartButton-Tabelle, in welcher hardcodierte oder regelbasierte Designvorgaben hinterlegt sind (z. B.: „Der Start-Button muss grün oder gräulich sein, darf aber nicht schwarz oder weiß sein, da die Schrift im Dark- bzw. White-Modus sonst unsichtbar wird“).

4. Rücksprung- und Feedback-Logik: Sobald der externe Tabellenschritt (wie die Prüfung des Start-Buttons) abgearbeitet ist, kehrt der Agent automatisch zur ursprünglichen Verhaltensreihenfolge zurück und führt die restlichen Schritte des Workflows aus.

5. agent_execution_memory: Dient als Erfahrungsspeicher und Feedback-Loop (Self-Reflection). Hier merkt sich das System, wie eine Aufgabe in einer anderen Tabelle bereits erfolgreich gelöst wurde, um den Knotenpunkt bei ähnlichen Anforderungen direkt wiederzuverbinden.

## Ordner Spezifizierung SQL Quelle

In [1]:
# Import
import sqlite3
import os

In [2]:
# Globale Definitionen für die Ordnerstruktur
ANKER_DIR = "Offline_AI"
BASE_DIR = "Knowledge"
AGENT_SUBDIR = "knowledge_agent_hierarchical_routing_tree_sql"
# globalisierte Variablen der SQLite-Datenbankdatei
DB_FILENAME = "knowledge_agent_routing_tree.db"

In [3]:
# def für die Initialisierung der Ordnerstruktur die SQLite-Datenbankdatei enthält
def initialize_find_folder():
    """
    Diese Funktion prüft, ob die notwendige Ordnerstruktur für den Knowledge Agent im Projekt vorhanden ist.
    Gibt bei jedem Teilschritt ein klares Status-Print in der Konsole aus.
    """
    print("--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---")

    # Schritt 0: Projekt-Root 'Offline_AI' ermitteln, um nicht im Notebooks-Ordner zu landen
    current_path = os.path.abspath(os.getcwd())
    print(f"--> [INFO] Start-Pfad des Notebooks: {current_path}")
    
    if ANKER_DIR.lower() in current_path.lower():
        base_parts = current_path.split(os.sep)
        anker_index = [p.lower() for p in base_parts].index(ANKER_DIR.lower())
        base_root = os.sep.join(base_parts[:anker_index + 1])
    else:
        base_root = current_path

    print(f"--> [ERFOLG] Projekt-Root '{ANKER_DIR}' identifiziert unter: {base_root}")

    # Schritt 1: Hauptverzeichnis "Knowledge" im Anker-Ordner suchen oder erstellen
    target_knowledge_dir = os.path.join(base_root, BASE_DIR)
    print(f"Suche nach Hauptverzeichnis: '{target_knowledge_dir}'...")
    if not os.path.exists(target_knowledge_dir):
        os.makedirs(target_knowledge_dir)
        print(f"--> [ERFOLG] Hauptverzeichnis '{target_knowledge_dir}' wurde neu erstellt.")
    else:
        print(f"--> [INFO] Hauptverzeichnis '{target_knowledge_dir}' wurde gefunden.")

    # Schritt 2: Unterordner für den Agenten im Knowledge-Verzeichnis suchen oder erstellen
    target_directory = os.path.join(target_knowledge_dir, AGENT_SUBDIR)
    print(f"Suche nach Unterordner: '{target_directory}'...")
    
    if not os.path.exists(target_directory):
        os.makedirs(target_directory)
        print(f"--> [ERFOLG] Unterordner '{AGENT_SUBDIR}' wurde neu erstellt.")
    else:
        print(f"--> [INFO] Unterordner '{AGENT_SUBDIR}' existiert bereits.")

    # Schritt 3: Vollständigen Pfad zur Datenbank zusammenbauen
    db_path = os.path.join(target_directory, DB_FILENAME)
    print(f"Pfad-Zusammenführung abgeschlossen. Zieldatei: '{db_path}'")
    print("--- [ENDE] Ordnerstruktur erfolgreich geprüft ---")
    
    return db_path

## Arbeitsweise Schrittverhalten
Um den Code modular, übersichtlich und sauber zu halten, folgen wir in diesem Jupyter Notebook einem strukturierten Ablauf. Die erforderlichen Imports werden grundsätzlich vor dem ersten Arbeits- und Schrittverhalten hinzugefügt.

Der fortlaufende Ablauf für jeden Schritt gestaltet sich wie folgt:

1. Globale Definitionen: Als Erstes definieren wir alle benötigten globalen Variablen und Konstanten als Code-Schnipsel.

2. Funktionserstellung (def): Danach erstellen wir eine neue Codezeile und implementieren die jeweilige Funktion (def), falls diese noch nicht existiert.

3. Ausführung & Visualisierung: Anschließend führen wir den Code aus und visualisieren den Erfolg oder Misserfolg direkt im Anschluss mit passenden print-Anweisungen.

4. Folgeschritte & Wiederverwendung: Falls der Arbeitsablauf weitere Teilschritte erfordert, wiederholen wir das Prinzip exakt passend zum jeweiligen Schritt:

- Zuerst die spezifischen globalen Variablen als neuer Code-Schnipsel.

- Das Einfügen der Funktion bzw. – falls der Code bereits existiert – das direkte Aufrufen des bestehenden Funktionsbegriffs (def).

- Die Ausführung des Codes inklusive der entsprechenden print-Erfolgsmeldung zur Validierung

## // agent_system_manifest
- Grundlegende Direktiven: Enthält die fundamentalen Systemanweisungen und operativen Richtlinien für die gesamte Architektur.

- Initialisierung: Das Large Language Model (LLM) liest diese Tabelle als allererstes aus, um den grundlegenden Kontext zu verstehen und zu wissen, wie es den nachfolgenden Routing-Baum zu navigieren hat.

In [4]:
# 1. GLOBALE VARIABLEN & SCHEMA-DEFINITIONEN
# Name der ersten Tabelle für das System-Manifest (Bedienungsanleitung für das LLM)
TABLE_MANIFEST = "agent_system_manifest"

# Globale Spaltennamen für diese Tabelle
COL_MANIFEST_KEY = "directive_key"          # Der eindeutige Schlüssel (Primary Key, z.B. 'core_logic')
COL_MANIFEST_VAL = "explanation_for_agent"    # Die eigentliche Anweisung / Erklärung für das LLM

In [5]:
# DEF Erstellt agent_system_manifest Tabelle in der SQLite-Datenbank
def create_system_manifest_table(db_path):
    """
    Erstellt die erste Tabelle ('agent_system_manifest') in der SQLite-Datenbank.
    Diese Tabelle dient dem LLM als reiner Aufklärungs- und Regel-Layer, bevor es mit der Suche beginnt.
    """
    print(f"--- [START] Erstelle Tabelle '{TABLE_MANIFEST}' ---")
    
    # 1. Verbindung zur SQLite-Datenbank herstellen (unter Verwendung des übergebenen Pfads)
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    print(f"--> [INFO] Verbindung zur Datenbank geöffnet: {db_path}")

    # 2. Tabelle erstellen unter Nutzung der globalen Konstanten
    create_table_query = f"""
        CREATE TABLE IF NOT EXISTS {TABLE_MANIFEST} (
            {COL_MANIFEST_KEY} TEXT PRIMARY KEY,
            {COL_MANIFEST_VAL} TEXT NOT NULL
        )
    """
    cursor.execute(create_table_query)
    print(f"--> [ERFOLG] Tabelle '{TABLE_MANIFEST}' wurde erfolgreich angelegt (oder war bereits vorhanden).")

    # 3. Metadaten-Direktiven zur reinen Aufklärung und Regeleinhaltung für das LLM
    initial_directives = [
        ("system_purpose", "This manifest defines the mandatory rules, security constraints, and operational boundaries for the LLM before any search begins."),
        ("immutable_base_rule", "CORE PROTECTION: Base tables without timestamps are strictly immutable. Deleting or overwriting core base tables is forbidden. Appends only."),
        ("versioning_and_registry", "ADMIN CONTROL: Always check the 'meta_admin_bootstrap_registry' where is_active = 1 to resolve active table versions and prevent table collisions."),
        ("navigation_handoff", "MANIFEST COMPLETE: After understanding these rules, transition to the indexed routing tree to evaluate user intent and select the correct execution path.")
    ]

    # 4. Daten sicher in die Tabelle schreiben (INSERT OR REPLACE verhindert Duplikate)
    insert_query = f"""
        INSERT OR REPLACE INTO {TABLE_MANIFEST} ({COL_MANIFEST_KEY}, {COL_MANIFEST_VAL}) 
        VALUES (?, ?)
    """
    cursor.executemany(insert_query, initial_directives)
    print("--> [ERFOLG] Aufklärungs- und Regel-Direktiven wurden erfolgreich in das Manifest eingefügt.")

    # 5. Transaktion speichern und Verbindung schließen
    conn.commit()
    conn.close()
    print(f"--- [ENDE] Tabelle '{TABLE_MANIFEST}' erfolgreich initialisiert ---")

In [6]:
# 2. DEF erste Ausführung der Initialisierung und Tabellen-Erstellung

# Schritt 1: Ordnerstruktur prüfen und den exakten Pfad zur Datenbank abrufen
db_path = initialize_find_folder()

# Schritt 2: System-Manifest Tabelle erstellen und füllen
create_system_manifest_table(db_path)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [ERFOLG] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' wurde neu erstellt.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--- [START] Erstelle Tabelle 'agent_system_manifest' ---
--> [INFO] Verbindung zur Datenban

## DEFINITIONEN GLOBAL

In [7]:
# DEF Erstellt create_routing_tree_table in der SQLite-Datenbank
def create_routing_tree_table(db_path):
    print(f"--- [START] Prüfe und initialisiere leere Tabelle '{TABLE_ROUTING}' ---")
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    print(f"--> [INFO] Verbindung zur Datenbank geöffnet: {db_path}")

    # Schritt 1: Prüfen, ob die Tabelle überhaupt existiert
    cursor.execute(f"""
        SELECT name FROM sqlite_master WHERE type='table' AND name='{TABLE_ROUTING}';
    """)
    table_exists = cursor.fetchone()

    if not table_exists:
        print(f"--> [INFO] Tabelle '{TABLE_ROUTING}' existiert noch nicht. Wird komplett neu erstellt...")
        columns_definition = ", ".join([f"{col_name} {col_type}" for col_name, col_type in ROUTING_COLUMNS.items()])
        create_table_query = f"""
            CREATE TABLE IF NOT EXISTS {TABLE_ROUTING} (
                {columns_definition}
            )
        """
        cursor.execute(create_table_query)
        print(f"--> [ERFOLG] Tabelle '{TABLE_ROUTING}' wurde erfolgreich und leer neu erstellt.")
    else:
        print(f"--> [INFO] Tabelle '{TABLE_ROUTING}' existiert bereits. Prüfe auf fehlende Spalten...")
        
        # Bestehende Spalten in der Datenbank auslesen
        cursor.execute(f"PRAGMA table_info({TABLE_ROUTING});")
        existing_columns_info = cursor.fetchall()
        existing_column_names = [col[1] for col in existing_columns_info]

        # Abgleich: Welche Spalten aus unserem Schema fehlen in der Datenbank?
        for col_name, col_type in ROUTING_COLUMNS.items():
            if col_name not in existing_column_names:
                print(f"--> [INFO] Spalte '{col_name}' fehlt in der Tabelle. Füge sie hinzu...")
                alter_query = f"ALTER TABLE {TABLE_ROUTING} ADD COLUMN {col_name} {col_type};"
                try:
                    cursor.execute(alter_query)
                    print(f"--> [ERFOLG] Spalte '{col_name}' erfolgreich hinzugefügt.")
                except Exception as e:
                    print(f"--> [FEHLER] Konnte Spalte '{col_name}' nicht hinzufügen: {e}")
            else:
                print(f"--> [OK] Spalte '{col_name}' ist bereits vorhanden.")

    conn.commit()
    conn.close()
    print(f"--- [ENDE] Leere Tabellen- und Spaltenprüfung für '{TABLE_ROUTING}' abgeschlossen ---")

In [8]:
# DEF Analyse der Tabellenstruktur und Ausgabe eines Python-Dictionaries mit Standardwerten
def analyze_table_structure(table_name, save_rite_status=True):
    """
    Analysiert eine SQLite-Tabelle, prüft deren Existenz und gibt ein passendes 
    Python-Dictionary (NAME_COLUMNS) mit Standardwerten und Format-Hinweisen aus.
    Fällt bei nicht existierenden Tabellen auf das globale ROUTING_COLUMNS-Schema zurück.
    """
    #print(f"--- [START] Analysiere Tabelle: '{table_name}' ---")
    
    # Exakten Pfad zur SQLite-Datenbank über die globale Ordnerstruktur-Funktion ermitteln
    db_path = initialize_find_folder()
    #print(f"--> [INFO] Ziel-Datenbank: {db_path}")
    #print(f"--> [INFO] Schreibschutz-Status (SAVE_RITE): {save_rite_status}")

    # Verbindung zur Datenbank herstellen
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    name_columns_result = {}

    try:
        # Spalteninformationen der global definierten Tabelle auslesen
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns_info = cursor.fetchall()
        
        # Fall 1: Tabelle existiert in der DB und hat Spalten
        if columns_info:
            #print(f"--> [ERFOLG] {len(columns_info)} Spalten direkt aus der DB-Tabelle '{table_name}' ausgelesen.")
            #print(" ")
            #print(" KOPIERE DIESEN BLOCK IN DAS NÄCHSTE CODEFELD UND FÜLLE IHN AUS:")
            #print(" ")
            print("NAME_ROWS= {")
            for col in columns_info:
                col_name = col[1]    # Spaltenname
                col_type = col[2]    # Datentyp (TEXT, INTEGER, etc.)
                
                # Format-Hinweise ableiten
                col_upper = col_type.upper()
                if "INT" in col_upper:
                    format_hint = "Format: Integer (Ganzzahl, z.B. 0, 1)"
                    default_val = 0
                elif "REAL" in col_upper:
                    format_hint = "Format: Float / Decimal (Dezimalzahl, z.B. 1.0)"
                    default_val = 0.0
                else:
                    format_hint = "Format: Text / String (z.B.Textbeschreibungen)"
                    default_val = ""
                    
                if col_name in ["year", "month", "day", "hour", "minute", "second"]:
                    format_hint = "Format: System-Auto-Fill (Wird automatisch vom System eingetragen, wenn leer)"

                print(f"    \"{col_name}\": {repr(default_val)},  # {format_hint}")
                name_columns_result[col_name] = default_val
                
            print("}")
            
        # Fall 2: Tabelle existiert noch nicht oder ist leer -> Nutzung des globalen Schemas (ROUTING_COLUMNS)
        else:
            #print(f"--> [HINWEIS] Die Tabelle '{table_name}' existiert noch nicht physisch in der DB.")
            #print("--> [INFO] Generiere Vorlage stattdessen aus dem globalen Python-Schema (ROUTING_COLUMNS)...")
            #print(" KOPIERE DIESEN BLOCK IN DAS NÄCHSTE CODEFELD UND FÜLLE IHN AUS:")
            #print(" ")
            print("NAME_ROWS= {")
            for col_name, col_definition in ROUTING_COLUMNS.items():
                col_upper = col_definition.upper()
                if "INT" in col_upper:
                    format_hint = "Format: Integer (Ganzzahl, z.B. 0, 1)"
                    default_val = 0
                elif "REAL" in col_upper:
                    format_hint = "Format: Float / Decimal (Dezimalzahl, z.B. 1.0)"
                    default_val = 0.0
                else:
                    format_hint = "Format: Text / String (z.B. Textbeschreibungen)"
                    default_val = ""
                    
                if col_name in ["year", "month", "day", "hour", "minute", "second"]:
                    format_hint = "Format: System-Auto-Fill (Wird automatisch vom System eingetragen, wenn leer)"

                print(f"    \"{col_name}\": {repr(default_val)},  # {format_hint}")
                name_columns_result[col_name] = default_val
                
            print("}")

    except Exception as e:
        print(f"--> [FEHLER] Konnte Spalten nicht auslesen: {e}")
    finally:
        conn.close()
        print(f"--- [ENDE] Analyse abgeschlossen ---")
        
    return #name_columns_result

In [9]:
# DEF Prüft und injiziert eine Batch von Routing-Zeilen in die Tabelle
import datetime

def insert_bootstrap_routing_rows(target_table_name, rows_data):
    """
    Checks if the table exists, validates rows, checks for existing records by node_id,
    compares values, and inserts missing rows or updates missing/different attributes intelligently.
    Includes auto-correction for single dictionary inputs (missing brackets []) and string-based row inputs.
    """
    print(f"--- [START] Checking and injecting row batch into table: '{target_table_name}' ---")
    
    # AUTO-KORREKTUR: Falls versehentlich ein einzelnes Dictionary statt einer Liste übergeben wurde
    if isinstance(rows_data, dict):
        print(f"--> [AUTO-CORRECT] Detected single dictionary in 'rows_data'. Automatically wrapped it into a list.")
        rows_data = [rows_data]
    elif not isinstance(rows_data, list):
        raise TypeError(f"Parameter 'rows_data' must be a list or a dictionary, got: {type(rows_data)}")

    db_path = initialize_find_folder()
    print(f"--> [INFO] Target Database: {db_path}")

    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cursor = conn.cursor()

    try:
        # 1. Check if the table exists
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name=?", (target_table_name,))
        if not cursor.fetchone():
            raise ValueError(f"Table '{target_table_name}' does not exist in the database!")

        # 2. Get table columns
        cursor.execute(f"PRAGMA table_info({target_table_name})")
        table_columns = {col[1] for col in cursor.fetchall()}

        # 3. Check current row count
        cursor.execute(f"SELECT COUNT(*) FROM {target_table_name}")
        table_row_count = cursor.fetchone()[0]
        print(f"--> [INFO] Current row count in '{target_table_name}': {table_row_count}")

        for item in rows_data:
            # AUTO-KORREKTUR: Falls ein String übergeben wurde
            if isinstance(item, str):
                print(f"----> [AUTO-CORRECT] Converted raw string item '{item}' into dictionary format.")
                node_data = {"node_id": item}
            elif isinstance(item, dict):
                node_data = item
            else:
                print(f"----> [ERROR] Skipping item due to unsupported data type: {type(item)}")
                continue

            node_id = node_data.get("node_id")
            if not node_id:
                print(f"----> [ERROR] Skipping node because 'node_id' is missing or empty in data: {node_data}")
                continue
                
            print(f"--> [PROCESSING] Inspecting Node ID: '{node_id}'")

            # Validate schema keys
            invalid_keys = [k for k in node_data.keys() if k not in table_columns]
            if invalid_keys:
                print(f"----> [SCHEMA WARNING] Unknown keys found (ignored): {invalid_keys}")

            # Auto-fill timestamps if empty or zero
            now = datetime.datetime.now()
            time_fields = {
                "year": now.year, "month": now.month, "day": now.day,
                "hour": now.hour, "minute": now.minute, "second": now.second
            }
            for field, val in time_fields.items():
                if field in node_data and (node_data[field] == 0 or not node_data[field]):
                    node_data[field] = val

            filtered_node_data = {k: v for k, v in node_data.items() if k in table_columns and v is not None}

            # SAVE_RITE status check
            save_rite_active = globals().get("SAVE_RITE", True)

            # 4. Check if the specific node already exists in the table
            cursor.execute(f"SELECT * FROM {target_table_name} WHERE node_id = ?", (node_id,))
            existing_row = cursor.fetchone()

            columns = list(filtered_node_data.keys())
            placeholders = ["?" for _ in columns]
            values = list(filtered_node_data.values())

            cols_joined = ", ".join(columns)
            placeholders_joined = ", ".join(placeholders)

            if not existing_row:
                # Case A: Node does not exist -> Insert all fields safely
                insert_query = f"""
                    INSERT INTO {target_table_name} ({cols_joined})
                    VALUES ({placeholders_joined})
                """
                try:
                    cursor.execute(insert_query, values)
                    print(f"----> [SUCCESS] Node '{node_id}' did not exist. Inserted completely.")
                except sqlite3.Error as ie:
                    print(f"----> [SQL ERROR DETAIL] Failed to insert Node ID '{node_id}' into '{target_table_name}'!")
                    print(f"----> [ERROR TEXT]: {ie}")
                    print(f"----> [PAYLOAD VALUES]: {filtered_node_data}")
                    raise ie
            else:
                # Case B: Node exists -> Compare values and check for missing/different attributes
                cursor.execute(f"PRAGMA table_info({target_table_name})")
                col_names = [col[1] for col in cursor.fetchall()]
                existing_data = dict(zip(col_names, existing_row))

                differences_found = False
                updates_needed = {}

                for key, new_val in filtered_node_data.items():
                    old_val = existing_data.get(key)
                    # If existing value is empty/null or different, mark for update
                    if old_val is None or old_val == "" or old_val == 0:
                        if new_val is not None and new_val != "":
                            updates_needed[key] = new_val
                            differences_found = True
                    elif old_val != new_val:
                        if not save_rite_active:
                            updates_needed[key] = new_val
                            differences_found = True
                        else:
                            print(f"----> [DIFFERENCE] Field '{key}' differs (Existing: '{old_val}' vs New: '{new_val}'). SAVE_RITE blocks overwrite.")

                if differences_found and updates_needed:
                    set_clauses = [f"{k} = ?" for k in updates_needed.keys()]
                    update_values = list(updates_needed.values())
                    update_values.append(node_id)

                    update_query = f"""
                        UPDATE {target_table_name} 
                        SET {", ".join(set_clauses)}
                        WHERE node_id = ?
                    """
                    try:
                        cursor.execute(update_query, update_values)
                        print(f"----> [UPDATED] Missing or allowed fields updated for Node '{node_id}': {list(updates_needed.keys())}")
                    except sqlite3.Error as ue:
                        print(f"----> [SQL ERROR DETAIL] Failed to update Node ID '{node_id}' in '{target_table_name}'!")
                        print(f"----> [ERROR TEXT]: {ue}")
                        raise ue
                else:
                    print(f"----> [IDENTICAL] Node '{node_id}' is already fully up-to-date. No changes needed.")

        conn.commit()
        print(f"--- [COMPLETE] Batch processing successfully finished for '{target_table_name}' ---")

    except Exception as e:
        print(f"--> [ERROR] Critical failure during batch processing on '{target_table_name}': {e}")
        conn.rollback()
        raise
    finally:
        conn.close()
        print(f"--- [END] Connection closed ---")

## Initialisierung der SQLite-Datenbank

Im folgenden Code-Block wird die Verbindung zur lokalen SQLite-Datenbank hergestellt und die oben beschriebene Tabellenstruktur fehlerfrei aufgebaut. Falls die Datenbank noch nicht existiert, wird sie automatisch generiert.

🛠️ Arbeitsweise & Schrittverhalten

1. Globale Tabellenspalten-Definition

Definition der globalen Konstanten und Spaltenstrukturen als Code-Schnipsel für die Ziel-Tabelle.

2. Initialisierung & Tabellenerstellung

Erstellung und Initialisierung der vordefinierten Tabelle in der SQLite-Datenbank.

3. Globale Schreibdefinition & Sicherheitskonfiguration

Definition des auszulesenden Tabellennamens sowie der Schutzparameter (z. B. overwrite=False), um automatisches Überschreiben zu verhindern.

4. Analyse der Tabellenstruktur

Auslesen der Tabellenstruktur basierend auf den globalen Vorgaben. Ausgabe als fertiges Python-Dictionary im Notebook-Output.

5. Vorgabe der Reihen

Kopieren des generierten Textbereichs, Vervollständigung und Hardcoding der einzelnen Datenzeilen für die automatisierte Injektion.

6. Sichere Injektion

Ausführung des geschützten Insert-Vorgangs. Bei doppelten IDs greift der Schutzmechanismus automatisch, sofern overwrite nicht explizit aktiviert wurde.

# Füge Tabellen und Rheien Hinzu

## // agent_routing_tree_indexed
fungiert als der zentrale Orchestrator (die Spitze der Pyramide) in einer agentenbasierten KI-Architektur. 
- Sie steuert die Entscheidungsrichtung und leitet eingehende Anfragen basierend auf dem erforderlichen Agenten-Verhalten an den passenden Fachbereich weiter, um eine präzise Verhaltensreihenfolge als Tabelle zu laden und diese anschließend strukturiert abzuarbeiten.

In [10]:
# 1. Globale Table Spalten Rite Definitionen 
TABLE_ROUTING = "agent_routing_tree_indexed"

ROUTING_COLUMNS = {
    # 1. Identifikation & Status
    "node_id": "TEXT PRIMARY KEY",              # Lokale ID innerhalb DIESER Weichentabelle (z.B. "000")
    "is_active": "INTEGER DEFAULT 1",           # Nur aktive Schritte (1) werden vom Agenten gelesen

    # 2. Zeitstempel & Protokollierung
    "year": "INTEGER",                          # Jahr der Ausführung/Erstellung
    "month": "INTEGER",                         # Monat
    "day": "INTEGER",                           # Tag
    "hour": "INTEGER",                          # Stunde
    "minute": "INTEGER",                        # Minute
    "second": "INTEGER",                        # Sekunde

    # 3. Inhaltliches Fundament & Spezifikation
    "topic_title": "TEXT",                      # Master-Muster  
    "topic_specification": "TEXT",              # Ausführliche Spezifikation

    # 4. Kognitives Verhalten & Ausführung
    "agent_instruction": "TEXT",                # Arbeitsanweisung Agent
    "example_code_snippet": "TEXT",             # Code-Schnipsel verbindung zur library_registry das tabellen mit fertigen mustern zur Kirurgische Übername im vorhaben besitzt
    "validation_rule": "TEXT",                  # Validierungsbedingung selbst Überprüfung der Eingaben und Ausgaben GEGENPRÜFUNG 

    # 5. Metriken & Zählung
    "execution_count": "INTEGER DEFAULT 0",     # Wie oft durchlaufen?
    "success_weight": "REAL DEFAULT 1.0",       # Gewichtung der Zuverlässigkeit bestätigung des ereichen des ziels

    # 6. Übergang in die nächste Tabelle
    "target_table": "TEXT",                     # NÄCHSTE TABELLE (Der Sprung in die Zieltabelle / Ausführungstabelle)
    "target_column_id": "TEXT",                 # Ziel-Spalte in der nächsten Tabelle
    "target_node_id": "TEXT"                    # Start-ID in der nächsten Tabelle (z.B. "000")
}
# 2. DEF erste Ausführung der Initialisierung und Tabellen-Erstellung
# Schritt 1: exakten Pfad zur bestehenden Datenbank abrufen
db_path = initialize_find_folder()

# Schritt 2: Die neue, Routing-Tabelle
create_routing_tree_table(db_path)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--- [START] Prüfe und initialisiere leere Tabelle 'agent_routing_tree_indexed' ---
--> [INFO] 

In [11]:
# 3. Globale Table Rite Definitionen 
# True verhindert das Überschreiben der Tabelle, wenn sie bereits existiert. False würde die Tabelle löschen und neu erstellen.
SAVE_RITE = True
TABLE_NAME = "agent_routing_tree_indexed"
# 4. Analyse der Tabellenstruktur
analyze_table_structure(TABLE_NAME, save_rite_status=SAVE_RITE)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
NAME_ROWS= {
    "node_id": '',  # Format: Text / String (z.B.Textbeschreibungen)
    "is_acti

In [12]:
# 5. Vorgabe der Reihen
NAME_ROWS = [
    {
        "node_id": "000",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "PROTECTED ADMIN INDEX",
        "topic_specification": "PROTECTED ADMIN INDEX: Points to the 'meta_admin_bootstrap_registry'. ACCESS DENIED if 'SAVE_RITE_ADMIN' is active! Defines strict rules for indexing new data to expand the library.",
        "agent_instruction": "DENY_ACCESS_IF_SAVE_WRITE_TRUE__ELSE_LOAD_META_ADMIN_REGISTRY",
        "example_code_snippet": "# ADMIN_BOOTSTRAP_ROUTING_SNIPPET",
        "validation_rule": "CHECK_SAVE_RITE_AND_ADMIN_PASSCODE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "meta_admin_bootstrap_registry",
        "target_column_id": "node_id",
        "target_node_id": "000",
    },
    {
        "node_id": "001",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "MANDATORY PRE-EXECUTION PLANNING",
        "topic_specification": "MANDATORY PRE-EXECUTION PLANNING: Forces the system to analyze the user request, verify existing database knowledge, and generate a complete execution blueprint before running any code or action.",
        "agent_instruction": "PARSE_INTENT__SYNTHESIZE_WORKFLOW__GENERATE_BLUEPRINT",
        "example_code_snippet": "# BLUEPRINT_GENERATOR_SNIPPET",
        "validation_rule": "CHECK_KNOWLEDGE_EXISTENCE__ELSE_TRIGGER_BLUEPRINT",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "pre_execution_blueprint_generator",
        "target_column_id": "node_id",
        "target_node_id": "000",
    },
    {
        "node_id": "ZZY",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "STANDARD LIBRARY REGISTRY",
        "topic_specification": "STANDARD LIBRARY REGISTRY: Provides structural rules, origin tracking, sequence steps, detailed descriptions, and ready-to-use Python code snippets for table execution.",
        "agent_instruction": "LOAD_LIBRARY_REGISTRY__APPLY_SEQUENCE_AND_CODE",
        "example_code_snippet": "# LIBRARY_REGISTRY_SNIPPET",
        "validation_rule": "CHECK_STANDARD_LIBRARY_REQUIREMENT",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "library_registry",
        "target_column_id": "node_id",
        "target_node_id": "000",
    },
    {
        "node_id": "ZZZ",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "GESCHÜTZTER NOTFALL-INDEX",
        "topic_specification": "GESCHÜTZTER NOTFALL-INDEX: Verweist auf die 'meta_admin_repair_registry'. Greift ein, wenn Kettenbrüche oder Fehler im Langzeitgedächtnis auftreten, und aktiviert das FX_CHAIN-Protokoll.",
        "agent_instruction": "ISOLATE_BROKEN_NODE__LOAD_REPAIR_REGISTRY__INIT_FX_CHAIN",
        "example_code_snippet": "# EMERGENCY_REPAIR_SNIPPET",
        "validation_rule": "CHECK_CHAIN_INTEGRITY_AND_TRIGGER_FX",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "meta_admin_repair_registry",
        "target_column_id": "node_id",
        "target_node_id": "000",
    }
]

In [13]:
# 6. Injektion der Rows in die Tabelle
insert_bootstrap_routing_rows(TABLE_NAME, NAME_ROWS)

--- [START] Checking and injecting row batch into table: 'agent_routing_tree_indexed' ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--> 

## // meta_admin_bootstrap_registry
🛡️ Die meta_admin_bootstrap_registry ist das zentrale Verzeichnis für den Agenten im administrativen Modus. Sie definiert die strikten Regeln und Arbeitsweisen für die Pflege aller Systemtabellen.

- Zeile 000 (Gesetzgebung & Verbotszone):
Enthält die fundamentalen Restriktionen. Hier liest der Agent aus, was er nicht tun darf (z. B. das absolute Löschverbot von Daten). Dies dient als unverrückbares Gesetzbuch vor jeder Aktion.

- Ab Zeile 001 (Tabellen-Verzeichnis & Arbeitsanweisung):
Listet alle existierenden Tabellen des Systems auf. Jede Zeile (z. B. 001, 002) stellt eine direkte Verbindung zu einer Zieltabelle her und liefert die exakte Anleitung, wie der Agent auf dieser speziellen Tabelle arbeiten darf (z. B. welche Spalten existieren und welche Daten eingepflegt werden dürfen), um eine korrekte und vollständige Datenpflege zu garantieren.

In [14]:
# 1. Globale Table Spalten Rite Definitionen 
TABLE_ROUTING = "meta_admin_bootstrap_registry"

ROUTING_COLUMNS = {
    # 1. Identifikation & Status
    "node_id": "TEXT PRIMARY KEY",              # Lokale ID innerhalb DIESER Weichentabelle (z.B. "000")
    "is_active": "INTEGER DEFAULT 1",           # Nur aktive Schritte (1) werden vom Agenten gelesen

    # 2. Zeitstempel & Protokollierung
    "year": "INTEGER",                          # Jahr der Ausführung/Erstellung
    "month": "INTEGER",                         # Monat
    "day": "INTEGER",                           # Tag
    "hour": "INTEGER",                          # Stunde
    "minute": "INTEGER",                        # Minute
    "second": "INTEGER",                        # Sekunde

    # 3. Inhaltliches Fundament & Spezifikation
    "topic_title": "TEXT",                      # Master-Muster  
    "topic_specification": "TEXT",              # Ausführliche Spezifikation

    # 4. Kognitives Verhalten & Ausführung
    "agent_instruction": "TEXT",                # Arbeitsanweisung Agent
    "example_code_snippet": "TEXT",             # Code-Schnipsel verbindung zur library_registry das tabellen mit fertigen mustern zur Kirurgische Übername im vorhaben besitzt
    "validation_rule": "TEXT",                  # Validierungsbedingung selbst Überprüfung der Eingaben und Ausgaben GEGENPRÜFUNG 

    # 5. Metriken & Zählung
    "execution_count": "INTEGER DEFAULT 0",     # Wie oft durchlaufen?
    "success_weight": "REAL DEFAULT 1.0",       # Gewichtung der Zuverlässigkeit bestätigung des ereichen des ziels

    # 6. Übergang in die nächste Tabelle
    "target_table": "TEXT",                     # NÄCHSTE TABELLE (Der Sprung in die Zieltabelle / Ausführungstabelle)
    "target_column_id": "TEXT",                 # Ziel-Spalte in der nächsten Tabelle
    "target_node_id": "TEXT"                    # Start-ID in der nächsten Tabelle (z.B. "000")
}
# 2. DEF erste Ausführung der Initialisierung und Tabellen-Erstellung
# Schritt 1: exakten Pfad zur bestehenden Datenbank abrufen
db_path = initialize_find_folder()

# Schritt 2: Die neue, Routing-Tabelle
create_routing_tree_table(db_path)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--- [START] Prüfe und initialisiere leere Tabelle 'meta_admin_bootstrap_registry' ---
--> [INF

In [15]:
# 3. Globale Table Rite Definitionen 
# True verhindert das Überschreiben der Tabelle, wenn sie bereits existiert. False würde die Tabelle löschen und neu erstellen.
SAVE_RITE = True
TABLE_NAME = "meta_admin_bootstrap_registry"
# 4. Analyse der Tabellenstruktur
analyze_table_structure(TABLE_NAME, save_rite_status=SAVE_RITE)


--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
NAME_ROWS= {
    "node_id": '',  # Format: Text / String (z.B.Textbeschreibungen)
    "is_acti

In [16]:
# 5. Vorgabe der Reihen
NAME_ROWS = [
    {
        "node_id": "000",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "MASTER GATEWAY TO TABOO RULES",
        "topic_specification": "MASTER_GATEWAY_TO_TABOO_RULES: Redirects the agent to the immutable governance and taboo rulebook before any admin or schema operation.",
        "agent_instruction": "MANDATORY REDIRECTION: Load the complete rulebook from meta_admin_taboo_rules (from node '000' to '999'). Absorb all prohibitions, non-destructive versioning mandates, and ensure the loop returns cleanly before altering any database structures.",
        "example_code_snippet": "# MASTER_GATEWAY_TABOO_ROUTING_SNIPPET",
        "validation_rule": "Must execute unconditionally as the very first step of any administrative workflow.",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "meta_admin_taboo_rules",
        "target_column_id": "node_id",
        "target_node_id": "000",
    }
]

In [17]:
# 6. Injektion der Rows in die Tabelle
insert_bootstrap_routing_rows(TABLE_NAME, NAME_ROWS)

--- [START] Checking and injecting row batch into table: 'meta_admin_bootstrap_registry' ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
-

## // meta_admin_taboo_rules
🚫 Die Tabelle meta_admin_taboo_rules repräsentiert die zuvor definierte 000-Tabelle. Sie enthält alle rollenspezifischen No-Go-Verhaltensweisen für den Agenten während des Admin-Modus (der Verwaltung und Pflege bereits bestehender Tabellen).

- Zweck: Speicherung der strikten Restriktionen, Verbote und unumstößlichen Gesetze, die der Agent als grundlegendes Schutzschild verinnerlicht, bevor er administrative Schreib- oder Strukturierungsaufgaben ausführt.

- Kernaufgabe: Ausschluss von Fehlverhalten (wie beispielsweise dem absoluten Löschverbot von Datensätzen oder Tabellen, welches ausschließlich dem menschlichen Benutzer vorbehalten ist), um die Integrität der gesamten Wissensbibliothek zu wahren.

In [18]:
# 1. Globale Table Spalten Rite Definitionen 
TABLE_ROUTING = "meta_admin_taboo_rules"

ROUTING_COLUMNS = {
    # 1. Identifikation & Status
    "node_id": "TEXT PRIMARY KEY",              # Lokale ID innerhalb DIESER Weichentabelle (z.B. "000")
    "is_active": "INTEGER DEFAULT 1",           # Nur aktive Schritte (1) werden vom Agenten gelesen

    # 2. Zeitstempel & Protokollierung
    "year": "INTEGER",                          # Jahr der Ausführung/Erstellung
    "month": "INTEGER",                         # Monat
    "day": "INTEGER",                           # Tag
    "hour": "INTEGER",                          # Stunde
    "minute": "INTEGER",                        # Minute
    "second": "INTEGER",                        # Sekunde

    # 3. Inhaltliches Fundament & Spezifikation
    "topic_title": "TEXT",                      # Master-Muster  
    "topic_specification": "TEXT",              # Ausführliche Spezifikation

    # 4. Kognitives Verhalten & Ausführung
    "agent_instruction": "TEXT",                # Arbeitsanweisung Agent
    "example_code_snippet": "TEXT",             # Code-Schnipsel verbindung zur library_registry das tabellen mit fertigen mustern zur Kirurgische Übername im vorhaben besitzt
    "validation_rule": "TEXT",                  # Validierungsbedingung selbst Überprüfung der Eingaben und Ausgaben GEGENPRÜFUNG 

    # 5. Metriken & Zählung
    "execution_count": "INTEGER DEFAULT 0",     # Wie oft durchlaufen?
    "success_weight": "REAL DEFAULT 1.0",       # Gewichtung der Zuverlässigkeit bestätigung des ereichen des ziels

    # 6. Übergang in die nächste Tabelle
    "target_table": "TEXT",                     # NÄCHSTE TABELLE (Der Sprung in die Zieltabelle / Ausführungstabelle)
    "target_column_id": "TEXT",                 # Ziel-Spalte in der nächsten Tabelle
    "target_node_id": "TEXT"                    # Start-ID in der nächsten Tabelle (z.B. "000")
}
# 2. DEF erste Ausführung der Initialisierung und Tabellen-Erstellung
# Schritt 1: exakten Pfad zur bestehenden Datenbank abrufen
db_path = initialize_find_folder()

# Schritt 2: Die neue, Routing-Tabelle
create_routing_tree_table(db_path)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--- [START] Prüfe und initialisiere leere Tabelle 'meta_admin_taboo_rules' ---
--> [INFO] Verb

In [19]:
# 3. Globale Table Rite Definitionen 
# True verhindert das Überschreiben der Tabelle, wenn sie bereits existiert. False würde die Tabelle löschen und neu erstellen.
SAVE_RITE = True
TABLE_NAME = "meta_admin_taboo_rules"
# 4. Analyse der Tabellenstruktur
analyze_table_structure(TABLE_NAME, save_rite_status=SAVE_RITE)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
NAME_ROWS= {
    "node_id": '',  # Format: Text / String (z.B.Textbeschreibungen)
    "is_acti

In [20]:
# 5. Vorgabe der Reihen
NAME_ROWS = [
    {
        "node_id": "000",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "SYSTEM_BOOT_AND_GOVERNANCE",
        "topic_specification": "LOAD_TABOO_MEMORY; Enforce absolute system governance and strict adherence to rule chains. Fallback action: FX_CHAIN.",
        "agent_instruction": "Execute system boot and governance protocols. Load taboo memory and enforce strict rule chain adherence.",
        "example_code_snippet": "# TABOO_BOOTSTRAP_SNIPPET_000",
        "validation_rule": "SYSTEM_BOOT_VALIDATED == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "meta_admin_taboo_rules",
        "target_column_id": "node_id",
        "target_node_id": "001",
    },
    {
        "node_id": "001",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "ANTI_CODE_SHRINKING",
        "topic_specification": "PRESERVE_CORE_LOGIC; Never delete, shrink, or remove existing baseline functions when writing code. Fallback action: FX_CHAIN.",
        "agent_instruction": "Ensure anti-code-shrinking governance is active. Preserve all core logic during updates.",
        "example_code_snippet": "# ANTI_SHRINK_SNIPPET_001",
        "validation_rule": "CORE_LOGIC_PRESERVED == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "meta_admin_taboo_rules",
        "target_column_id": "node_id",
        "target_node_id": "002",
    },
    {
        "node_id": "002",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "ADDITIVE_MODIFICATION_ONLY",
        "topic_specification": "APPLY_ADDITIVE_UPDATES; Always introduce new capabilities as new versions or extensions without destroying past logic. Fallback action: FX_CHAIN.",
        "agent_instruction": "Verify that all changes follow an strictly additive modification protocol.",
        "example_code_snippet": "# ADDITIVE_MOD_SNIPPET_002",
        "validation_rule": "ADDITIVE_MODE_VERIFIED == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "meta_admin_taboo_rules",
        "target_column_id": "node_id",
        "target_node_id": "003",
    },
    {
        "node_id": "003",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "PROHIBIT_SCHEMA_DROPS",
        "topic_specification": "NO_DROP_OR_TRUNCATE; Never execute DROP TABLE, DROP DATABASE, or TRUNCATE commands on core system tables. Fallback action: FX_CHAIN.",
        "agent_instruction": "Enforce strict prohibition against schema drops or table truncations.",
        "example_code_snippet": "# NO_DROP_SNIPPET_003",
        "validation_rule": "NO_DESTRUCTIVE_DDL_DETECTED == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "meta_admin_taboo_rules",
        "target_column_id": "node_id",
        "target_node_id": "004",
    },
    {
        "node_id": "004",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "PROHIBIT_HARD_DELETES",
        "topic_specification": "NO_HARD_DELETE; Never use physical DELETE statements on historical or core configuration records; use versioning and is_active=0 instead. Fallback action: FX_CHAIN.",
        "agent_instruction": "Enforce soft deletion policies via versioning and is_active flags instead of hard deletes.",
        "example_code_snippet": "# NO_HARD_DELETE_SNIPPET_004",
        "validation_rule": "SOFT_DELETE_PROTOCOL_ACTIVE == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "meta_admin_taboo_rules",
        "target_column_id": "node_id",
        "target_node_id": "005",
    },
    {
        "node_id": "005",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "PROHIBIT_UNVALIDATED_SCHEMA_ALTERATION",
        "topic_specification": "NO_BLIND_ALTER; Never modify existing table columns or data types without verifying backward compatibility and schema integrity hashes. Fallback action: FX_CHAIN.",
        "agent_instruction": "Block unvalidated schema alterations and verify backward compatibility checks.",
        "example_code_snippet": "# NO_BLIND_ALTER_SNIPPET_005",
        "validation_rule": "SCHEMA_COMPATIBILITY_CHECKED == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "meta_admin_taboo_rules",
        "target_column_id": "node_id",
        "target_node_id": "006",
    },
    {
        "node_id": "006",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "PROHIBIT_INFINITE_LOOPS",
        "topic_specification": "NO_RECURSIVE_LOCK; Never create circular dependency routing or self-referencing loops that bypass the FX_CHAIN recovery mechanism. Fallback action: FX_CHAIN.",
        "agent_instruction": "Ensure routing integrity and prevent circular locks that bypass recovery mechanisms.",
        "example_code_snippet": "# NO_LOOP_SNIPPET_006",
        "validation_rule": "ROUTING_GRAPH_ACYCLIC == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "meta_admin_taboo_rules",
        "target_column_id": "node_id",
        "target_node_id": "999",
    },
    {
        "node_id": "999",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "LOOP_TERMINATOR_AND_RETURN",
        "topic_specification": "EXIT_TABOO_CHAIN; Finalize memory loading and safely return execution control to the main administration registry. Fallback action: FX_CHAIN.",
        "agent_instruction": "Finalize taboo rule loading and safely transition back to the administration registry bootstrap row.",
        "example_code_snippet": "# TERMINATE_TABOO_SNIPPET_999",
        "validation_rule": "TABOO_RULES_FULLY_LOADED == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "meta_admin_bootstrap_registry",
        "target_column_id": "node_id",
        "target_node_id": "001",
    }
]

In [21]:
# 6. Injektion der Rows in die Tabelle
insert_bootstrap_routing_rows(TABLE_NAME, NAME_ROWS)

--- [START] Checking and injecting row batch into table: 'meta_admin_taboo_rules' ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--> [INF

## // meta_admin_repair_registry
🛠️ meta_admin_repair_registry (ZZZ-Ebene)
- Zweck: Zentrales Repositorium für sämtliche Instandsetzungs- und Fallback-Mechanismen zur automatisierten Selbstreparatur der Wissensbibliothek.

- Inhalt: Tabellen und Code-Logiken für Fehler- und Ausnahmesituationen (z. B. fehlerhafte Sequenzen, abgebrochene Sprünge).

- Funktionsweise: Definiert die exakte Auslösung von Fallback-Begriffen.

Ablauf: Fürt zu unterschiedlichen Tabellen, Gibt dem Agenten eine Schritt-für-Schritt-Reihenfolge vor, wie Fehler behoben und Instandsetzungen anschliessend nachkontrolliert werden.

In [22]:
# 1. Globale Table Spalten Rite Definitionen 
TABLE_ROUTING = "meta_admin_repair_registry"

ROUTING_COLUMNS = {
    # 1. Identifikation & Status
    "node_id": "TEXT PRIMARY KEY",              # Lokale ID innerhalb DIESER Weichentabelle (z.B. "000")
    "is_active": "INTEGER DEFAULT 1",           # Nur aktive Schritte (1) werden vom Agenten gelesen

    # 2. Zeitstempel & Protokollierung
    "year": "INTEGER",                          # Jahr der Ausführung/Erstellung
    "month": "INTEGER",                         # Monat
    "day": "INTEGER",                           # Tag
    "hour": "INTEGER",                          # Stunde
    "minute": "INTEGER",                        # Minute
    "second": "INTEGER",                        # Sekunde

    # 3. Inhaltliches Fundament & Spezifikation
    "topic_title": "TEXT",                      # Master-Muster  
    "topic_specification": "TEXT",              # Ausführliche Spezifikation

    # 4. Kognitives Verhalten & Ausführung
    "agent_instruction": "TEXT",                # Arbeitsanweisung Agent
    "example_code_snippet": "TEXT",             # Code-Schnipsel verbindung zur library_registry das tabellen mit fertigen mustern zur Kirurgische Übername im vorhaben besitzt
    "validation_rule": "TEXT",                  # Validierungsbedingung selbst Überprüfung der Eingaben und Ausgaben GEGENPRÜFUNG 

    # 5. Metriken & Zählung
    "execution_count": "INTEGER DEFAULT 0",     # Wie oft durchlaufen?
    "success_weight": "REAL DEFAULT 1.0",       # Gewichtung der Zuverlässigkeit bestätigung des ereichen des ziels

    # 6. Übergang in die nächste Tabelle
    "target_table": "TEXT",                     # NÄCHSTE TABELLE (Der Sprung in die Zieltabelle / Ausführungstabelle)
    "target_column_id": "TEXT",                 # Ziel-Spalte in der nächsten Tabelle
    "target_node_id": "TEXT"                    # Start-ID in der nächsten Tabelle (z.B. "000")
}
# 2. DEF erste Ausführung der Initialisierung und Tabellen-Erstellung
# Schritt 1: exakten Pfad zur bestehenden Datenbank abrufen
db_path = initialize_find_folder()

# Schritt 2: Die neue, Routing-Tabelle
create_routing_tree_table(db_path)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--- [START] Prüfe und initialisiere leere Tabelle 'meta_admin_repair_registry' ---
--> [INFO] 

In [23]:
# 3. Globale Table Rite Definitionen 
# True verhindert das Überschreiben der Tabelle, wenn sie bereits existiert. False würde die Tabelle löschen und neu erstellen.
SAVE_RITE = True
TABLE_NAME = "meta_admin_repair_registry"
# 4. Analyse der Tabellenstruktur
analyze_table_structure(TABLE_NAME, save_rite_status=SAVE_RITE)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
NAME_ROWS= {
    "node_id": '',  # Format: Text / String (z.B.Textbeschreibungen)
    "is_acti

In [24]:
# 5. Vorgabe der Reihen
NAME_ROWS = [
    {
        "node_id": "000",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "Primary bridge head into FX Chain",
        "topic_specification": "Primary bridge head from Repair Registry into the FX Chain sequence table. Transfer control flow directly to the FX_CHAIN table structure without intermediate fallback loops.",
        "agent_instruction": "EXECUTE_PYRAMID_DESCENT; Verify agent alignment and authorize clean handoff to FX_CHAIN.",
        "example_code_snippet": "# FX_CHAIN_BRIDGE_SNIPPET_000",
        "validation_rule": "AGENT_ALIGNMENT_VERIFIED == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "FX_CHAIN",
        "target_column_id": "node_id",
        "target_node_id": "000",
    }
]

In [25]:
# 6. Injektion der Rows in die Tabelle
insert_bootstrap_routing_rows(TABLE_NAME, NAME_ROWS)

--- [START] Checking and injecting row batch into table: 'meta_admin_repair_registry' ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--> 

## // FX_CHAIN
🔗 FX_CHAIN
- Zweck: Die primäre Instandsetzungs- und Reparaturkette für Sequenz- und Reihenfolgefehler (z. B. fehlerhafte Sprünge zwischen Schritten wie 001 zu 002 oder unsaubere Direkt-Abbrüche zu 999).

- Fehlererkennung: Registriert, wenn hinzugefügte Tabellenzeilen nicht sauber und eindeutig chronologisch verknüpft sind (z. B. veraltete Versionen oder gescheiterte Logik-Pfade während der Admin-Bearbeitung).

- Auslösung & Parameter: Benötigt zwingend die Information, welche konkrete Tabelle den Fehler ausgelöst hat, um den genauen Ursprung zu identifizieren.

Ablauf & Neustart: Startet einen Korrektur-Agenten, der die fehlerhafte Reihenfolge in der betroffenen Tabelle bereinigt und repariert. Im Anschluss führt der FX_Chain den Fallback zurück zur Haupttabelle aus, damit die ursprüngliche Benutzeranfrage vollautomatisch von Anfang an neu gestartet und wie gewollt abgearbeitet wird.

In [26]:
# 1. Globale Table Spalten Rite Definitionen 
TABLE_ROUTING = "FX_CHAIN"

ROUTING_COLUMNS = {
    # 1. Identifikation & Status
    "node_id": "TEXT PRIMARY KEY",              # Lokale ID innerhalb DIESER Weichentabelle (z.B. "000")
    "is_active": "INTEGER DEFAULT 1",           # Nur aktive Schritte (1) werden vom Agenten gelesen

    # 2. Zeitstempel & Protokollierung
    "year": "INTEGER",                          # Jahr der Ausführung/Erstellung
    "month": "INTEGER",                         # Monat
    "day": "INTEGER",                           # Tag
    "hour": "INTEGER",                          # Stunde
    "minute": "INTEGER",                        # Minute
    "second": "INTEGER",                        # Sekunde

    # 3. Inhaltliches Fundament & Spezifikation
    "topic_title": "TEXT",                      # Master-Muster  
    "topic_specification": "TEXT",              # Ausführliche Spezifikation

    # 4. Kognitives Verhalten & Ausführung
    "agent_instruction": "TEXT",                # Arbeitsanweisung Agent
    "example_code_snippet": "TEXT",             # Code-Schnipsel verbindung zur library_registry das tabellen mit fertigen mustern zur Kirurgische Übername im vorhaben besitzt
    "validation_rule": "TEXT",                  # Validierungsbedingung selbst Überprüfung der Eingaben und Ausgaben GEGENPRÜFUNG 

    # 5. Metriken & Zählung
    "execution_count": "INTEGER DEFAULT 0",     # Wie oft durchlaufen?
    "success_weight": "REAL DEFAULT 1.0",       # Gewichtung der Zuverlässigkeit bestätigung des ereichen des ziels

    # 6. Übergang in die nächste Tabelle
    "target_table": "TEXT",                     # NÄCHSTE TABELLE (Der Sprung in die Zieltabelle / Ausführungstabelle)
    "target_column_id": "TEXT",                 # Ziel-Spalte in der nächsten Tabelle
    "target_node_id": "TEXT"                    # Start-ID in der nächsten Tabelle (z.B. "000")
}
# 2. DEF erste Ausführung der Initialisierung und Tabellen-Erstellung
# Schritt 1: exakten Pfad zur bestehenden Datenbank abrufen
db_path = initialize_find_folder()

# Schritt 2: Die neue, Routing-Tabelle
create_routing_tree_table(db_path)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--- [START] Prüfe und initialisiere leere Tabelle 'FX_CHAIN' ---
--> [INFO] Verbindung zur Dat

In [27]:
# 3. Globale Table Rite Definitionen 
# True verhindert das Überschreiben der Tabelle, wenn sie bereits existiert. False würde die Tabelle löschen und neu erstellen.
SAVE_RITE = True
TABLE_NAME = "FX_CHAIN"
# 4. Analyse der Tabellenstruktur
analyze_table_structure(TABLE_NAME, save_rite_status=SAVE_RITE)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
NAME_ROWS= {
    "node_id": '',  # Format: Text / String (z.B.Textbeschreibungen)
    "is_acti

In [28]:
# 5. Vorgabe der Reihen
NAME_ROWS = [
    {
        "node_id": "000",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "INIT LOOP AND IDENTIFY BROKEN TABLE",
        "topic_specification": "DYNAMIC_CAPTURE_FROM_SYSTEM: Übergibt dynamisch den Namen der fehlerhaften Tabelle an den Agenten. Initialisiert den Reparatur-Scope für die fehlerhafte Tabelle.",
        "agent_instruction": "START LOOP: Read the 'triggering_table' where the sequence break occurred. Establish the operational scope for table-wide node iteration.",
        "example_code_snippet": """import sqlite3\ndef capture_error(db_path, failed_table):\n    # Initialisiert den Reparatur-Scope für die fehlerhafte Tabelle\n    pass""",
        "validation_rule": "triggering_table IS NOT NULL AND table_exists(triggering_table) == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "fx_chain",
        "target_column_id": "node_id",
        "target_node_id": "001",
    },
    {
        "node_id": "001",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "ITERATE EACH NODE AND CHECK ACTIVE",
        "topic_specification": "INHERIT_FROM_000: Ermittelt alle eindeutigen Node-IDs in der fehlerhaften Tabelle zur Überprüfung des Status.",
        "agent_instruction": "WHILE nodes remain unverified: Select the current node_id (starting from '000'). Check all rows matching this node_id. Evaluate 'is_active' (1 or 0) and parse the precise timestamp (year, month, day, hour, minute, second).",
        "example_code_snippet": """def scan_unique_nodes(cursor, target_table):\n    # Ermittelt alle eindeutigen Node-IDs in der fehlerhaften Tabelle\n    cursor.execute(f'SELECT DISTINCT node_id FROM {target_table} ORDER BY node_id ASC')\n    return [row[0] for row in cursor.fetchall()]""",
        "validation_rule": "current_node_scanned == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "fx_chain",
        "target_column_id": "node_id",
        "target_node_id": "002",
    },
    {
        "node_id": "002",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "COMPARE TIMESTAMPS AND RESOLVE DUPLICATES",
        "topic_specification": "INHERIT_FROM_000: Vergleicht Zeitstempel, setzt den neuesten Eintrag auf is_active=1 und ältere Duplikate auf is_active=0.",
        "agent_instruction": "FOR duplicates of the same node_id: Compare their timestamps. Deactivate older entries by setting 'is_active = 0'. Keep only the absolute newest entry active by setting 'is_active = 1'. If a node has no active entry at all, reactivate its latest valid version.",
        "example_code_snippet": """def resolve_duplicates_and_timestamps(cursor, target_table, current_node):\n    # Vergleicht Zeitstempel: Setzt den neuesten Eintrag auf is_active=1, ältere Duplikate auf is_active=0\n    cursor.execute(f'SELECT id, year, month, day, hour, minute, second FROM {target_table} WHERE node_id = ?', (current_node,))\n    rows = cursor.fetchall()\n    sorted_rows = sorted(rows, key=lambda r: (r[1], r[2], r[3], r[4], r[5], r[6]), reverse=True)\n    newest_id = sorted_rows[0][0]\n    cursor.execute(f'UPDATE {target_table} SET is_active = 1 WHERE id = ?', (newest_id,))\n    older_ids = [r[0] for r in sorted_rows[1:]]\n    if older_ids:\n        placeholders = ','.join(['?'] * len(older_ids))\n        cursor.execute(f'UPDATE {target_table} SET is_active = 0 WHERE id IN ({placeholders})', older_ids)""",
        "validation_rule": "exactly_one_active_row_per_node_id == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "fx_chain",
        "target_column_id": "node_id",
        "target_node_id": "003",
    },
    {
        "node_id": "003",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "CHECK LOOP COMPLETION",
        "topic_specification": "INHERIT_FROM_000: Überprüft die lückenlose logische Reihenfolge und steuert den Schleifenrücksprung.",
        "agent_instruction": "CHECK LOOP CONDITION: Are there any unverified node steps remaining in the table? If YES, loop back to node_id '001' for the next sequence index. If NO (all rows from start to end are scanned and cleanly linked), proceed to node '999'.",
        "example_code_snippet": """def verify_loop_and_sequence(cursor, target_table):\n    # Überprüft die lückenlose logische Reihenfolge und Parent-Verknüpfungen\n    pass""",
        "validation_rule": "all_table_nodes_fully_validated == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "fx_chain",
        "target_column_id": "node_id",
        "target_node_id": "999",
    },
    {
        "node_id": "999",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "TERMINATE LOOP AND RESTART USER INTENT",
        "topic_specification": "INHERIT_FROM_000: Beendet die Reparatur, gibt Kontrolle zurück an das Hauptregister und startet den Workflow neu.",
        "agent_instruction": "TERMINATE REPAIR LOOP: The sequence table is now completely healed, synchronized, and free of contradictions. Exit the fallback loop, jump back to the main routing registry, and re-execute the user's complete original workflow from index '000' onward.",
        "example_code_snippet": """def terminate_and_restart():\n    # Beendet die Reparatur und gibt Kontrolle zurück an das Hauptregister\n    pass""",
        "validation_rule": "table_chain_integrity_verified AND return_to_main_registry_authorized == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "meta_admin_bootstrap_registry",
        "target_column_id": "node_id",
        "target_node_id": "000",
    }
]

In [29]:
# 6. Injektion der Rows in die Tabelle
insert_bootstrap_routing_rows(TABLE_NAME, NAME_ROWS)

--- [START] Checking and injecting row batch into table: 'FX_CHAIN' ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--> [INFO] Target Data

## // 🧠 pre_execution_blueprint_generator

- Definiert das Fundament, wie das System Benutzeranfragen fehlerfrei verarbeitet, den Kernel scannt, eigenständig plant, im Hintergrund ressourcenschonend arbeitet und über ein strukturiertes Pyramiden_rückschleife-Netzwerk mit Fehler- und Rücksprung-Codes (999) vollautomatisch korrigiert und steuert.

1. Kognitive Planungssteuerung: Fungiert als zentraler, autonomer Steuerungs- und Logikkern, der Roheanfragen abfängt, in saubere Ausführungsstrukturen übersetzt und das gesamte Vorhaben lückenlos durchplant(user_query), bevor Code ausgeführt wird.

2. Kernel-Scan & Bibliotheken-Registry: Scannt den Python-Kernel alle 24h neu auf existierende Importe und Werkzeuge, die zur Planung und zum Coding genutzt werden können. Erstellt daraus eine dedizierte Bibliothekstabelle (kernel_tool), wobei jeder Eintrag mit einem Zeitstempel versehen wird, falls diese nicht existiert. Verschwundene Module werden aus Gründen der Historie niemals gelöscht, sondern lediglich auf is_active = FALSE gesetzt, während neue Tools ergänzt werden.

3. Ressourcen- & Leistungsüberwachung:(hardware_enviroment) Überprüft die Leistung des Rechners und nutzt konsequent ein absolutes Minimum nach Best Practices an Prozessorleistung und Systemressourcen, um das Vorhaben im Hintergrund zu starten und auszuführen.

4. Anfrage-Erfassung & Token-Optimierung: Nutzt intelligente Tools zur Erfassung der Benutzeranfrage, versteht das exakte Vorhaben des Benutzers und verbessert die Aussage durch Token-Reduzierung, Schärfung und zielgerichtete Formulierungen und tragt es wieder in (user_query).

5. Präzisierung & Multiple-Choice-Dialog: Stellt gezielte Fragen an den Benutzer (inklusive präziser Beispielsaussagen als Multiple-Choice-Optionen), um das Vorhaben zu spezifizieren, falls Unklarheiten bestehen. Bei unklaren Wiederholungen vom Benutzer macht das System eine Internetrecherche. Bei klaren, eindeutigen Äußerungen wird direkt und ohne Umwege mit dem Vorhaben fortgeschritten.

6. Resiliente Hintergrund-Ausarbeitung (Stromausfallsicherheit): Arbeitet das Projekt im Hintergrund mit geringer Belastung, kognitiven Pausen (time.sleep) und permanenter Protokollierung ab. Dank Checkpoints (gespeicherte Indizes nach jedem Teilschritt) ist das System absolut stromausfallsicher – bricht der Strom oder das System ein, arbeitet es nahtlos dort weiter, wo es stehen geblieben ist.

7. Best-Practice-Integration: Plant Funktionen und Code-Strukturen unter Einbindung aktueller Best Practices aus dem Internet (sofern eine Internetverbindung besteht) und baut das Projekt inkrementell wie ein Skelett auf Liest und Legt def in der forgesehene (library_registry) in der passenden tabelle ein für zukünftige beschleunigung mit fertigen code oder mustern.

8. JARVIS-Prinzip & Anwesenheits-Manager: Überprüft bei Abschluss der Planung die Anwesenheit des Benutzers. Sobald eine Interaktion stattfindet, spricht das System den aktuellen Stand der Entwicklung vor, informiert über noch offene Punkte oder laufende Hintergrundprozesse und koordiniert die nächsten Schritte. Tritt während der Arbeit ein Fehler auf, greift das Pyramiden_rückschleife-Netzwerk über den Fehlercode 999, korrigiert den Pfad vollautomatisch über den FX-Agenten und führt den Prozess sauber in den Hauptstrang zurück.

9. Pyramiden-Wissens-Schleife (Zeilen-Sub-Tabellen & rekursive Loops): Jede einzelne Ausführungszeile bzw. Planungsstufe verfügt über eine eigene, dedizierte Untertabelle (z. B. step_knowledge_loops), die als eigenständiger Wissens-Loop fungiert. Aktuelle Software-Architektur-Best-Practices für hierarchische Agenten-Zustandsmaschinen (Hierarchical State Machines) und rekursive Multi-Agenten-Loops (RecursiveMAS) zeigen, dass die Aufteilung in isolierte Teilschritt-Tabellen den Token-Overhead drastisch reduziert und die Stabilität erhöht.

    - Funktionsweise des Zeilen-Loops: Jede Tabellenzeile sammelt ihren eigenen Kontext, importiert die spezifischen Kernel-Bibliotheken für diesen Teilbereich, führt lokale Validierungen durch und kommuniziert über den standardisierten Rücksprung-Code 999 (mit bis zu 999 differenzierten Sub-Codes wie 999.1 für Syntax, 999.2 für Import-Fehler).
    - Rückführung in den Hauptstrang: Sobald ein Zeilen-Loop seine Aufgabe erfolgreich beendet oder nach einem Fehler durch den FX-Agenten über den 999-Pfad bereinigt wurde, übergibt er das Ergebnis an die übergeordnete Pyramiden-Ebene zurück. Dadurch wird sichergestellt, dass jede Zeile unabhängig voneinander lernt, wächst und sich selbstständig optimiert, ohne den globalen Zustand zu korrumpieren.

In [30]:
# 1. Globale Table Spalten Rite Definitionen 
TABLE_ROUTING = "pre_execution_blueprint_generator"

ROUTING_COLUMNS = {
    # 1. Identifikation & Status
    "node_id": "TEXT PRIMARY KEY",              # Lokale ID innerhalb DIESER Weichentabelle (z.B. "000")
    "is_active": "INTEGER DEFAULT 1",           # Nur aktive Schritte (1) werden vom Agenten gelesen

    # 2. Zeitstempel & Protokollierung
    "year": "INTEGER",                          # Jahr der Ausführung/Erstellung
    "month": "INTEGER",                         # Monat
    "day": "INTEGER",                           # Tag
    "hour": "INTEGER",                          # Stunde
    "minute": "INTEGER",                        # Minute
    "second": "INTEGER",                        # Sekunde

    # 3. Inhaltliches Fundament & Spezifikation
    "topic_title": "TEXT",                      # Master-Muster  
    "topic_specification": "TEXT",              # Ausführliche Spezifikation

    # 4. Kognitives Verhalten & Ausführung
    "agent_instruction": "TEXT",                # Arbeitsanweisung Agent
    "example_code_snippet": "TEXT",             # Code-Schnipsel verbindung zur library_registry das tabellen mit fertigen mustern zur Kirurgische Übername im vorhaben besitzt
    "validation_rule": "TEXT",                  # Validierungsbedingung selbst Überprüfung der Eingaben und Ausgaben GEGENPRÜFUNG 

    # 5. Metriken & Zählung
    "execution_count": "INTEGER DEFAULT 0",     # Wie oft durchlaufen?
    "success_weight": "REAL DEFAULT 1.0",       # Gewichtung der Zuverlässigkeit bestätigung des ereichen des ziels

    # 6. Übergang in die nächste Tabelle
    "target_table": "TEXT",                     # NÄCHSTE TABELLE (Der Sprung in die Zieltabelle / Ausführungstabelle)
    "target_column_id": "TEXT",                 # Ziel-Spalte in der nächsten Tabelle
    "target_node_id": "TEXT"                    # Start-ID in der nächsten Tabelle (z.B. "000")
}
# 2. DEF erste Ausführung der Initialisierung und Tabellen-Erstellung
# Schritt 1: exakten Pfad zur bestehenden Datenbank abrufen
db_path = initialize_find_folder()

# Schritt 2: Die neue, Routing-Tabelle
create_routing_tree_table(db_path)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--- [START] Prüfe und initialisiere leere Tabelle 'pre_execution_blueprint_generator' ---
--> 

In [31]:
# 3. Globale Table Rite Definitionen 
# True verhindert das Überschreiben der Tabelle, wenn sie bereits existiert. False würde die Tabelle löschen und neu erstellen.
SAVE_RITE = True
TABLE_NAME = "pre_execution_blueprint_generator"
# 4. Analyse der Tabellenstruktur
analyze_table_structure(TABLE_NAME, save_rite_status=SAVE_RITE)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
NAME_ROWS= {
    "node_id": '',  # Format: Text / String (z.B.Textbeschreibungen)
    "is_acti

In [32]:
# 5. Vorgabe der Reihen
NAME_ROWS = [
    {
        "node_id": "000",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "INIT COGNITIVE PLANNING AND CAPTURE",
        "topic_specification": "INIT_COGNITIVE_PLANNING_AND_CAPTURE: Intercept incoming raw user requests. Translate them into clean operational structures and plan the full execution pipeline before code generation begins.",
        "agent_instruction": "START SYSTEM: Intercept incoming raw user requests. Translate them into clean operational structures and plan the full execution pipeline before code generation begins.",
        "example_code_snippet": """def capture_and_plan_query(raw_query):\n    # Initialize the planning core for the raw user query\n    cleaned_query = raw_query.strip().lower()\n    return {"status": "initialized", "query": cleaned_query}""",
        "validation_rule": "raw_query IS NOT NULL AND planning_scope_established == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "pre_execution_blueprint_generator",
        "target_column_id": "node_id",
        "target_node_id": "001",
    },
    {
        "node_id": "001",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "KERNEL SCAN AND REGISTRY UPDATE",
        "topic_specification": "KERNEL_SCAN_AND_REGISTRY_UPDATE: Scan the Python kernel for existing imports and tools every 24h. Build or update 'python_kernel_library_registry'. Deactivate missing modules by setting 'is_active = 0' instead of deleting them.",
        "agent_instruction": "SCAN KERNEL: Scan the Python kernel for existing imports and tools every 24h. Build or update 'python_kernel_library_registry'. Deactivate missing modules by setting 'is_active = 0' instead of deleting them.",
        "example_code_snippet": """import importlib.util\ndef scan_kernel_modules(registry_db, module_list):\n    # Scan available Python modules in the kernel\n    for mod in module_list:\n        exists = importlib.util.find_spec(mod) is not None\n        # Update registry logic here\n        pass""",
        "validation_rule": "kernel_registry_updated AND missing_modules_deactivated == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "pre_execution_blueprint_generator",
        "target_column_id": "node_id",
        "target_node_id": "002",
    },
    {
        "node_id": "002",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "RESOURCE AND PERFORMANCE MONITORING",
        "topic_specification": "RESOURCE_AND_PERFORMANCE_MONITORING: Check system performance and ensure minimal CPU and resource consumption following best practices for background execution.",
        "agent_instruction": "MONITOR RESOURCES: Check system performance and ensure minimal CPU and resource consumption following best practices for background execution.",
        "example_code_snippet": """import psutil\ndef check_system_resources():\n    # Monitor CPU and RAM usage for resource-efficient operation\n    cpu_usage = psutil.cpu_percent(interval=1)\n    return cpu_usage < 80""",
        "validation_rule": "system_resources_within_safe_limits == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "pre_execution_blueprint_generator",
        "target_column_id": "node_id",
        "target_node_id": "003",
    },
    {
        "node_id": "003",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "QUERY CAPTURE AND TOKEN OPTIMIZATION",
        "topic_specification": "QUERY_CAPTURE_AND_TOKEN_OPTIMIZATION: Capture the user request, reduce token overhead, and sharpen formulations for precise execution.",
        "agent_instruction": "OPTIMIZE TOKENS: Capture the user request, reduce token overhead, and sharpen formulations for precise execution.",
        "example_code_snippet": """def optimize_user_query(raw_text):\n    # Reduce token overhead and sharpen query formulation\n    tokens = raw_text.split()\n    optimized = ' '.join([t for t in tokens if len(t) > 2])\n    return optimized""",
        "validation_rule": "query_token_optimized == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "pre_execution_blueprint_generator",
        "target_column_id": "node_id",
        "target_node_id": "004",
    },
    {
        "node_id": "004",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "CLARIFICATION AND MULTIPLE CHOICE DIALOG",
        "topic_specification": "CLARIFICATION_AND_MULTIPLE_CHOICE_DIALOG: If ambiguities exist, ask targeted questions with multiple-choice examples. Trigger web research if unclarities persist. Proceed directly if the query is clear.",
        "agent_instruction": "CLARIFY INTENT: If ambiguities exist, ask targeted questions with multiple-choice examples. Trigger web research if unclarities persist. Proceed directly if the query is clear.",
        "example_code_snippet": """def evaluate_clarity(user_intent_score):\n    # Check whether follow-up questions or web research are required\n    if user_intent_score < 0.8:\n        return "trigger_multiple_choice_dialog"\n    return "proceed_direct_execution\"""",
        "validation_rule": "intent_clearly_resolved OR dialog_initiated == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "pre_execution_blueprint_generator",
        "target_column_id": "node_id",
        "target_node_id": "005",
    },
    {
        "node_id": "005",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "RESILIENT BACKGROUND EXECUTION AND CHECKPOINTS",
        "topic_specification": "RESILIENT_BACKGROUND_EXECUTION_AND_CHECKPOINTS: Run project execution in the background using low load, cognitive pauses (time.sleep), and permanent checkpointing for power outage resilience.",
        "agent_instruction": "EXECUTE BACKGROUND: Run project execution in the background using low load, cognitive pauses (time.sleep), and permanent checkpointing for power outage resilience.",
        "example_code_snippet": """import time\ndef background_step_runner(step_func, *args):\n    # Execute sub-steps with cognitive pauses and checkpoints\n    time.sleep(0.5)\n    result = step_func(*args)\n    return result""",
        "validation_rule": "checkpoint_saved_successfully == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "pre_execution_blueprint_generator",
        "target_column_id": "node_id",
        "target_node_id": "006",
    },
    {
        "node_id": "006",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "BEST PRACTICE SKELETON INTEGRATION",
        "topic_specification": "BEST_PRACTICE_SKELETON_INTEGRATION: Plan code structures using current online best practices and build the project incrementally like a skeleton.",
        "agent_instruction": "INTEGRATE BEST PRACTICES: Plan code structures using current online best practices and build the project incrementally like a skeleton.",
        "example_code_snippet": """def build_incremental_skeleton(module_blueprint):\n    # Build the project incrementally following best practices\n    skeleton = [step for step in module_blueprint]\n    return skeleton""",
        "validation_rule": "best_practices_embedded == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "pre_execution_blueprint_generator",
        "target_column_id": "node_id",
        "target_node_id": "007",
    },
    {
        "node_id": "007",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "JARVIS PRESENCE MANAGER AND ERROR FX",
        "topic_specification": "JARVIS_PRESENCE_MANAGER_AND_ERROR_FX: Verify user presence upon completion. Announce current development status. If errors occur, trigger error code 999 to auto-correct via the FX agent.",
        "agent_instruction": "CHECK PRESENCE & ERRORS: Verify user presence upon completion. Announce current development status. If errors occur, trigger error code 999 to auto-correct via the FX agent.",
        "example_code_snippet": """def jarvis_manager_audit(error_code, user_present):\n    # Audit presence and control the 999 error correction network\n    if error_code == 999:\n        return "trigger_fx_agent_repair"\n    return "report_status_to_user" if user_present else "continue_background\"""",
        "validation_rule": "audit_completed_and_errors_routed == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "pre_execution_blueprint_generator",
        "target_column_id": "node_id",
        "target_node_id": "999",
    },
    {
        "node_id": "999",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "PYRAMID KNOWLEDGE LOOP SUB ROUTING",
        "topic_specification": "PYRAMID_KNOWLEDGE_LOOP_SUB_ROUTING: Isolate sub-steps into dedicated sub-tables ('step_knowledge_loops') for RecursiveMAS execution. Communicate via sub-codes (999.1, 999.2) and return cleaned results to the main thread.",
        "agent_instruction": "PYRAMID LOOP HANDOFF: Isolate sub-steps into dedicated sub-tables ('step_knowledge_loops') for RecursiveMAS execution. Communicate via sub-codes (999.1, 999.2) and return cleaned results to the main thread.",
        "example_code_snippet": """def pyramid_sub_loop_handler(sub_code, sub_table):\n    # Manage isolated sub-step loops and return results\n    if sub_code.startswith("999."):\n        # Execute specific correction logic\n        pass\n    return "merged_to_main_thread\"""",
        "validation_rule": "sub_loop_successfully_merged == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "meta_admin_bootstrap_registry",
        "target_column_id": "node_id",
        "target_node_id": "000",
    }
]

In [33]:
# 6. Injektion der Rows in die Tabelle
insert_bootstrap_routing_rows(TABLE_NAME, NAME_ROWS)

--- [START] Checking and injecting row batch into table: 'pre_execution_blueprint_generator' ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft -

## 📚 // library_registry

- Grundlegende Direktiven: Enthält die verbindlichen strukturellen Regeln, Herkunftsverknüpfungen, thematischen Spezifikationen und fertigen Code-Schnipsel für neu zu generierende oder auszuführende Tabellen.

- Initialisierung: Das Large Language Model (LLM) greift auf diese Bibliothek zu, um exakte Arbeitsanweisungen und chirurgisch genaue Code-Muster abzurufen, Halluzinationen zu verhindern  und Fehler über den dokumentierten Soll-Zustand eigenständig zu analysieren und zu korrigieren.

In [34]:
# 1. Globale Table Spalten Definitionen (Exakt nach visuellem Ablauf)
TABLE_ROUTING = "library_registry"

ROUTING_COLUMNS = {
    # 1. Identifikation & Status
    "node_id": "TEXT PRIMARY KEY",              # Lokale ID innerhalb DIESER Tabelle
    "is_active": "INTEGER DEFAULT 1",           # Nur aktive Schritte (1) werden vom Agenten gelesen

    # 2. Zeitstempel & Protokollierung
    "year": "INTEGER",                          # Jahr der Ausführung/Erstellung
    "month": "INTEGER",                         # Monat
    "day": "INTEGER",                           # Tag
    "hour": "INTEGER",                          # Stunde
    "minute": "INTEGER",                        # Minute
    "second": "INTEGER",                        # Sekunde

    # 3. Inhaltliches Fundament & Spezifikation
    "topic_title": "TEXT NOT NULL",             # Kurze Bezeichnung für den Fokus
    "topic_specification": "TEXT NOT NULL",     # Vollständiger Erklärungstext (Soll-Zustand & Basis für Fehleranalyse/Reparatur)

    # 4. Kognitives Verhalten & Ausführung
    "agent_instruction": "TEXT NOT NULL",       # Arbeitsanweisung für den Agenten
    "example_code_snippet": "TEXT NOT NULL",    # Chirurgisch genauer Code-Schnipsel (verhindert Halluzinationen)
    "validation_rule": "TEXT NOT NULL",         # Bedingung, die zur Erfüllung vorliegen muss

    # 5. Metriken & Zählung
    "execution_count": "INTEGER DEFAULT 0",     # Wie oft durchlaufen?
    "success_weight": "REAL DEFAULT 1.0",       # Gewichtung der Zuverlässigkeit

    # 6. Übergang in die nächste Tabelle
    "target_table": "TEXT",                     # Name der nächsten Zieltabelle
    "target_column_id": "TEXT",                 # Ziel-Spalte in der nächsten Tabelle
    "target_node_id": "TEXT"                    # Ziel-Knoten/Zeile in der nächsten Tabelle
}

In [35]:
# 2. DEF erste Ausführung der Initialisierung und Tabellen-Erstellung
# Schritt 1: exakten Pfad zur bestehenden Datenbank abrufen
db_path = initialize_find_folder()

# Schritt 2: Die neue, Routing-Tabelle
create_routing_tree_table(db_path)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--- [START] Prüfe und initialisiere leere Tabelle 'library_registry' ---
--> [INFO] Verbindung

In [36]:
# 3. Globale Table Rite Definitionen 
# True verhindert das Überschreiben der Tabelle, wenn sie bereits existiert. False würde die Tabelle löschen und neu erstellen.
SAVE_RITE = True
TABLE_NAME = "library_registry"
# 4. Analyse der Tabellenstruktur
analyze_table_structure(TABLE_NAME, save_rite_status=SAVE_RITE)


--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
NAME_ROWS= {
    "node_id": '',  # Format: Text / String (z.B.Textbeschreibungen)
    "is_acti

In [37]:
# 5. Vorgabe der Reihen (Knoten 000 für die Erstellung/Verbindung der mandatory Tabelle)
NAME_ROWS = [
    {
        "node_id": "000",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "Mandatory Library Bootstrapping & Core Routing Setup",
        "topic_specification": "Initialisiert das Pyramidenkonzept der Bibliothek, indem die verbindliche 'mandatory_column_mapping' und die Grundstruktur der 'standard_library_registry' validiert und im System als primäre Steuerungsebene verankert werden.",
        "agent_instruction": "CREATE & CONNECT MANDATORY TABLE: Initialize the mandatory library table structure based on the schema mapping. Ensure all required columns (node_id, is_active, timestamps, topic definitions, and target routing) are properly linked to establish the root of the pyramid architecture.",
        "example_code_snippet": """import sqlite3\n\ndef initialize_mandatory_registry(db_connection, table_name, schema_columns):\n    # Erstellt oder verbindet die mandatory Bibliothek als Basis des Pyramidenkonzepts\n    cursor = db_connection.cursor()\n    columns_def = ", ".join([f"{col} {datatype}" for col, datatype in schema_columns.items()])\n    cursor.execute(f"CREATE TABLE IF NOT EXISTS {table_name} ({columns_def});")\n    db_connection.commit()\n    return f"Table {table_name} successfully initialized and connected." """,
        "validation_rule": "mandatory_table_exists AND primary_key_node_id_active == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "standard_column_library_registry",
        "target_column_id": "node_id",
        "target_node_id": "000"
    },
    {
        "node_id": "001",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "Python Kernel Function Library (defs)",
        "topic_specification": "Speichert und verwaltet wiederverwendbare Programmier-Logik, mathematische Operationen, Datenverarbeitungs-Skripte und lokale Hilfsfunktionen als standardisierte 'def'-Blöcke.",
        "agent_instruction": "REGISTER FUNCTIONS: Maintain the core repository of clean Python functions ('defs'). Ensure every tool or snippet has a clear signature, parameters, and return value to prevent redundant code generation.",
        "example_code_snippet": """def compute_data_metric(value_list):\n    # Reiner Code-Baustein für lokale Berechnungen\n    return sum(value_list) / len(value_list) if value_list else 0.0""",
        "validation_rule": "function_signature_valid AND python_syntax_correct == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "def_library_registry",
        "target_column_id": "node_id",
        "target_node_id": "000"
    },
    {
        "node_id": "002",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "Agent Toolkit & External Tool Routing (Web/AI)",
        "topic_specification": "Definiert Musteragenten für externe Operationen. Reghlt, wann eine Internetverbindung genutzt wird, ob eine Google-Suche erfolgt, oder ob eine sekundäre (kostenfreie) KI für Text- oder Bildgenerierungen dazwischengeschaltet wird.",
        "agent_instruction": "EXECUTE EXTERNAL TOOL: Evaluate whether the task requires local execution or an external tool call. If needed, trigger the specific pattern (e.g., Web-Search, API query to a secondary free AI model, or image generation) via standardized wrapper functions.",
        "example_code_snippet": """import requests\n\ndef query_external_tool(tool_type, prompt):\n    # Steuert den externen Abruf (z.B. Google-Suche, KI-Zwischenschaltung oder kostenfreier API-Endpoint)\n    if tool_type == "web_search":\n        # Logik für Web-Abruf\n        pass\n    elif tool_type == "secondary_ai":\n        # Logik für die Abfrage einer anderen kostenfreien KI\n        pass\n    return {"status": "success", "source": tool_type}""",
        "validation_rule": "external_tool_reachable AND correct_wrapper_selected == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "toolkit_library_registry",
        "target_column_id": "node_id",
        "target_node_id": "000"
    }
]

In [38]:
# 6. Injektion der Rows in die Tabelle
insert_bootstrap_routing_rows(TABLE_NAME, NAME_ROWS)

--- [START] Checking and injecting row batch into table: 'library_registry' ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--> [INFO] Tar

## 📚 // standard_column_library_registry
- Zweck: Fester Mindest-Tabellenaufbau für neue Tabellen nach dem Pyramidenprinzip (chronologische Verbindung).

    - Fixe Standards (Unveränderbar): Kernelemente wie node_id und is_active sowie die Übergangs-Knoten (Wer bin ich / Wo gehe ich hin) dürfen niemals durch andere Begriffe oder Synonyme ersetzt werden. Die exakte Reihenfolge bleibt starr.

    - Erweiterung: Zusätzliche Spalten für neue Notwendigkeiten werden ausschließlich vor oder nach dem kognitiven Verhalten eingefügt.

    - Ziel: Schutz der Funktionsfähigkeit, damit bestehende Abläufe, Verbindungen zur Vor-/Nachatabelle und die Integrität des Systems nicht überschrieben oder verschoben werden.

In [39]:
# 1. Globale Table Spalten Definitionen (Exakt nach visuellem Ablauf)
TABLE_ROUTING = "standard_column_library_registry"

ROUTING_COLUMNS = {
    # 1. Identifikation & Status
    "node_id": "TEXT PRIMARY KEY",              # Lokale ID innerhalb DIESER Tabelle
    "is_active": "INTEGER DEFAULT 1",           # Nur aktive Schritte (1) werden vom Agenten gelesen

    # 2. Zeitstempel & Protokollierung
    "year": "INTEGER",                          # Jahr der Ausführung/Erstellung
    "month": "INTEGER",                         # Monat
    "day": "INTEGER",                           # Tag
    "hour": "INTEGER",                          # Stunde
    "minute": "INTEGER",                        # Minute
    "second": "INTEGER",                        # Sekunde

    # 3. Inhaltliches Fundament & Spezifikation
    "topic_title": "TEXT NOT NULL",             # Kurze Bezeichnung für den Fokus
    "topic_specification": "TEXT NOT NULL",     # Vollständiger Erklärungstext (Soll-Zustand & Basis für Fehleranalyse/Reparatur)

    # 4. Kognitives Verhalten & Ausführung
    "agent_instruction": "TEXT NOT NULL",       # Arbeitsanweisung für den Agenten
    "example_code_snippet": "TEXT NOT NULL",    # Chirurgisch genauer Code-Schnipsel (verhindert Halluzinationen)
    "validation_rule": "TEXT NOT NULL",         # Bedingung, die zur Erfüllung vorliegen muss

    # 5. Metriken & Zählung
    "execution_count": "INTEGER DEFAULT 0",     # Wie oft durchlaufen?
    "success_weight": "REAL DEFAULT 1.0",       # Gewichtung der Zuverlässigkeit

    # 6. Übergang in die nächste Tabelle
    "target_table": "TEXT",                     # Name der nächsten Zieltabelle
    "target_column_id": "TEXT",                 # Ziel-Spalte in der nächsten Tabelle
    "target_node_id": "TEXT"                    # Ziel-Knoten/Zeile in der nächsten Tabelle
}

In [40]:
# 2. DEF erste Ausführung der Initialisierung und Tabellen-Erstellung
# Schritt 1: exakten Pfad zur bestehenden Datenbank abrufen
db_path = initialize_find_folder()

# Schritt 2: Die neue, Routing-Tabelle
create_routing_tree_table(db_path)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--- [START] Prüfe und initialisiere leere Tabelle 'standard_column_library_registry' ---
--> [

In [41]:
# 3. Globale Table Rite Definitionen 
# True verhindert das Überschreiben der Tabelle, wenn sie bereits existiert. False würde die Tabelle löschen und neu erstellen.
SAVE_RITE = True
TABLE_NAME = "standard_column_library_registry"
# 4. Analyse der Tabellenstruktur
analyze_table_structure(TABLE_NAME, save_rite_status=SAVE_RITE)


--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
NAME_ROWS= {
    "node_id": '',  # Format: Text / String (z.B.Textbeschreibungen)
    "is_acti

In [42]:
# 5. Vorgabe der Reihen (Knoten 000 für die Erstellung/Verbindung der mandatory Tabelle)
NAME_ROWS = [
    {
        "node_id": "000",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "Mandatory Library Bootstrapping & Core Routing Setup",
        "topic_specification": "Initialisiert das Pyramidenkonzept der Bibliothek, indem die verbindliche 'mandatory_column_mapping' und die Grundstruktur der 'standard_library_registry' validiert und im System als primäre Steuerungsebene verankert werden.",
        "agent_instruction": "CREATE & CONNECT MANDATORY TABLE: Initialize the mandatory library table structure based on the schema mapping. Ensure all required columns (node_id, is_active, timestamps, topic definitions, and target routing) are properly linked to establish the root of the pyramid architecture.",
        "example_code_snippet": """import sqlite3\n\ndef initialize_mandatory_registry(db_connection, table_name, schema_columns):\n    # Erstellt oder verbindet die mandatory Bibliothek als Basis des Pyramidenkonzepts\n    cursor = db_connection.cursor()\n    columns_def = ", ".join([f"{col} {datatype}" for col, datatype in schema_columns.items()])\n    cursor.execute(f"CREATE TABLE IF NOT EXISTS {table_name} ({columns_def});")\n    db_connection.commit()\n    return f"Table {table_name} successfully initialized and connected." """,
        "validation_rule": "mandatory_table_exists AND primary_key_node_id_active == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "standard_column_library_registry",
        "target_column_id": "node_id",
        "target_node_id": "001"
    },
    {
        "node_id": "001",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "Python Kernel Function Library (defs)",
        "topic_specification": "Speichert und verwaltet wiederverwendbare Programmier-Logik, mathematische Operationen, Datenverarbeitungs-Skripte und lokale Hilfsfunktionen als standardisierte 'def'-Blöcke.",
        "agent_instruction": "REGISTER FUNCTIONS: Maintain the core repository of clean Python functions ('defs'). Ensure every tool or snippet has a clear signature, parameters, and return value to prevent redundant code generation.",
        "example_code_snippet": """def compute_data_metric(value_list):\n    # Reiner Code-Baustein für lokale Berechnungen\n    return sum(value_list) / len(value_list) if value_list else 0.0""",
        "validation_rule": "function_signature_valid AND python_syntax_correct == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "standard_column_library_registry",
        "target_column_id": "node_id",
        "target_node_id": "002"
    },
    {
        "node_id": "002",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "Agent Toolkit & External Tool Routing (Web/AI)",
        "topic_specification": "Definiert Musteragenten für externe Operationen. Regelt, wann eine Internetverbindung genutzt wird, ob eine Google-Suche erfolgt, oder ob eine sekundäre (kostenfreie) KI für Text- oder Bildgenerierungen dazwischengeschaltet wird.",
        "agent_instruction": "EXECUTE EXTERNAL TOOL: Evaluate whether the task requires local execution or an external tool call. Reference the centralized defs in Python Kernel Function Library where applicable. If needed, trigger the specific pattern via standardized wrapper functions.",
        "example_code_snippet": """# REF: python_kernel_function_library -> query_external_tool\nimport requests\n\ndef query_external_tool(tool_type, prompt):\n    # Steuert den externen Abruf über zentrale Bibliotheksdefinition\n    if tool_type == "web_search":\n        pass\n    elif tool_type == "secondary_ai":\n        pass\n    return {"status": "success", "source": tool_type}""",
        "validation_rule": "external_tool_reachable AND correct_wrapper_selected == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "standard_column_library_registry",
        "target_column_id": "node_id",
        "target_node_id": "003"
    },
    {
        "node_id": "999",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "Step finish return - Chain Termination & Fallback",
        "topic_specification": "Signalisiert das erfolgreiche Ende der aktuellen Ausführungskette. Löst den Fallback zurück zur übergeordneten Master-Planung aus, damit der Agent nahtlos mit den restlichen, noch offenen Abläufen fortfahren kann.",
        "agent_instruction": "TERMINATE & FALLBACK: Verify that all preceding steps are executed and validated. Disengage the current tool chain, release active states, and return control back to the primary orchestration loop / master planning to process remaining scheduled tasks.",
        "example_code_snippet": """# REF: master_planning_fallback\ndef terminate_and_fallback(current_chain_id):\n    # Beendet die aktuelle Kette und übergibt die Kontrolle zurück an die Master-Planung\n    print(f"Chain {current_chain_id} finished successfully. Resuming master plan...")\n    return {"status": "chain_terminated", "fallback": "master_planning_active"}""",
        "validation_rule": "chain_completed AND fallback_to_master_successful == TRUE",
        "execution_count": 0,
        "success_weight": 1.0,
        "target_table": "standard_column_library_registry",
        "target_column_id": "node_id",
        "target_node_id": "000"
    }
]

In [43]:
# 6. Injektion der Rows in die Tabelle
insert_bootstrap_routing_rows(TABLE_NAME, NAME_ROWS)

--- [START] Checking and injecting row batch into table: 'standard_column_library_registry' ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft --

## 📚 // def_library_registry
- Zweck: Zentraler Speicher für wiederverwendbare Programmier-Logik und standardisierte def-Blöcke, um Code-Redundanz über verschiedene Projekte hinweg zu vermeiden.

- Ziel: Schutz der Modularität und Wiederverwendbarkeit, damit fertige Code-Snippets zentral gepflegt und von anderen Tabellen oder Agenten-Schritten sauber referenziert werden können.

- 📌 Feld-Spezifikationen für den Agenten
    - topic_title (Eindeutiger Titel): Hier wird der präzise Funktionsname eingetragen (z. B. verify_and_inject_rows), damit sofort ersichtlich ist, welche spezifische Routine dahintersteckt.

    - topic_specification (Funktionserklärung): Eine klare Beschreibung, was der Code macht, wie er funktioniert, warum er eingesetzt wird und welches Ergebnis er liefert.

    - agent_instruction (Handlungsanweisung): Die klare Anweisung an den Agenten, das Snippet als Vorlage zu verwenden und chronologisch einzubinden.

    - example_code_snippet (Vollständige Def): Hier wird die vollständige, einsatzbereite Python-Funktion (inklusive Imports und korrekter Zeilenumbrüche via \n) eins zu eins als zusammenhängender Code-Block hinterlegt, damit der Agent sie direkt übernehmen und ausführen kann.

    - validation_rule & success_weight: Die logische Überprüfungsregel (z. B. Syntax- und Signatur-Check) sowie die Planungs-Gewichtung (z. B. 1.0), um die fehlerfreie Integrität des Snippets sicherzustellen.

In [44]:
# 1. Globale Table Spalten Definitionen (Exakt nach visuellem Ablauf)
TABLE_ROUTING = "def_library_registry"

ROUTING_COLUMNS = {
    # 1. Identifikation & Status
    "node_id": "TEXT PRIMARY KEY",              # Lokale ID innerhalb DIESER Tabelle
    "is_active": "INTEGER DEFAULT 1",           # Nur aktive Schritte (1) werden vom Agenten gelesen

    # 2. Zeitstempel & Protokollierung
    "year": "INTEGER",                          # Jahr der Ausführung/Erstellung
    "month": "INTEGER",                         # Monat
    "day": "INTEGER",                           # Tag
    "hour": "INTEGER",                          # Stunde
    "minute": "INTEGER",                        # Minute
    "second": "INTEGER",                        # Sekunde

    # 3. Inhaltliches Fundament & Spezifikation
    "topic_title": "TEXT NOT NULL",             # Kurze Bezeichnung für den Fokus
    "topic_specification": "TEXT NOT NULL",     # Vollständiger Erklärungstext (Soll-Zustand & Basis für Fehleranalyse/Reparatur)

    # 4. Kognitives Verhalten & Ausführung
    "agent_instruction": "TEXT NOT NULL",       # Arbeitsanweisung für den Agenten
    "example_code_snippet": "TEXT NOT NULL",    # Chirurgisch genauer Code-Schnipsel (verhindert Halluzinationen)
    "validation_rule": "TEXT NOT NULL",         # Bedingung, die zur Erfüllung vorliegen muss

    # 5. Metriken & Zählung
    "execution_count": "INTEGER DEFAULT 0",     # Wie oft durchlaufen?
    "success_weight": "REAL DEFAULT 1.0",       # Gewichtung der Zuverlässigkeit

    # 6. Übergang in die nächste Tabelle
    "target_table": "TEXT",                     # Name der nächsten Zieltabelle
    "target_column_id": "TEXT",                 # Ziel-Spalte in der nächsten Tabelle
    "target_node_id": "TEXT"                    # Ziel-Knoten/Zeile in der nächsten Tabelle
}

In [45]:
# 2. DEF erste Ausführung der Initialisierung und Tabellen-Erstellung
# Schritt 1: exakten Pfad zur bestehenden Datenbank abrufen
db_path = initialize_find_folder()

# Schritt 2: Die neue, Routing-Tabelle
create_routing_tree_table(db_path)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--- [START] Prüfe und initialisiere leere Tabelle 'def_library_registry' ---
--> [INFO] Verbin

In [46]:
# 3. Globale Table Rite Definitionen 
# True verhindert das Überschreiben der Tabelle, wenn sie bereits existiert. False würde die Tabelle löschen und neu erstellen.
SAVE_RITE = True
TABLE_NAME = "def_library_registry"
# 4. Analyse der Tabellenstruktur
analyze_table_structure(TABLE_NAME, save_rite_status=SAVE_RITE)


--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
NAME_ROWS= {
    "node_id": '',  # Format: Text / String (z.B.Textbeschreibungen)
    "is_acti

In [47]:
# 5. Vorgabe der Reihen (Knoten 000 für die Erstellung/Verbindung der mandatory Tabelle)
NAME_ROWS = [
    {
        "node_id": "000",
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
        "topic_title": "verify_and_inject_rows",
        "topic_specification": "Überprüft die Datenbankschema-Struktur einer SQLite-Tabelle anhand erwarteter Spalten und injiziert strukturierte Datensätze zeilenweise (mit Upsert-Logik) inklusive Protokollierung.",
        "agent_instruction": "USE AS TEMPLATE: Execute this code snippet to verify schema columns and inject structured rows sequentially into the target SQLite registry table using db_path, table_name, schema_columns, and name_rows.",
        "example_code_snippet": "import sqlite3\n\ndef verify_and_inject_rows(db_path, table_name, schema_columns, name_rows):\n    conn = sqlite3.connect(db_path)\n    cursor = conn.cursor()\n    \n    columns_def = \", \".join([f\"{col} {datatype}\" for col, datatype in schema_columns.items()])\n    cursor.execute(f\"CREATE TABLE IF NOT EXISTS {table_name} ({columns_def});\")\n    \n    cursor.execute(f\"PRAGMA table_info({table_name});\")\n    existing_columns = [row[1] for row in cursor.fetchall()]\n    expected_columns = list(schema_columns.keys())\n    \n    for i, expected in enumerate(expected_columns):\n        if i >= len(existing_columns) or existing_columns[i] != expected:\n            raise ValueError(f\"SCHEMA-FEHLER bei Spalte {i+1}. Erwartet: '{expected}'\")\n\n    for row in name_rows:\n        cols = \", \".join(row.keys())\n        placeholders = \", \".join([\"?\"] * len(row))\n        values = tuple(row.values())\n        \n        sql = f\"INSERT OR REPLACE INTO {table_name} ({cols}) VALUES ({placeholders})\"\n        cursor.execute(sql, values)\n        print(f\"-> Node '{row.get('node_id')}' erfolgreich injiziert (Nächster Ziel-Node: {row.get('target_node_id')})\")\n        \n    conn.commit()\n    conn.close()\n    print(\"--- [ERFOLG] Alle Schritte chronologisch in der Tabelle hinterlegt. ---\")",
        "validation_rule": "function_signature_valid AND sqlite_execution_successful == TRUE",
        "execution_count": 1,
        "success_weight": 1.0,
        "target_table": "def_library_registry",
        "target_column_id": "node_id",
        "target_node_id": "000"
    }
]

In [48]:
# 6. Injektion der Rows in die Tabelle
insert_bootstrap_routing_rows(TABLE_NAME, NAME_ROWS)

--- [START] Checking and injecting row batch into table: 'def_library_registry' ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--> [INFO]